In [1]:
# Close any open handles to the file
try:
    ds.close()
except: pass

# # Save
# ds = xr.Dataset({
#     "TOT_PREC": da_all,
#     "hourly_rain": hourly_rain,
# })
# ds.to_netcdf("icon_ch1_TOT_PREC_all_lead_times.nc")
# print("Saved!")

In [6]:
import requests
r = requests.get("https://data.geo.admin.ch/api/stac/v1/collections")
cols = r.json()
inca = [c["id"] for c in cols["collections"] if "inca" in c["id"].lower()]
print(inca)  # will likely be empty

[]


In [ ]:
import xarray as xr
import numpy as np

da = xr.open_dataarray("icon_ch1_TOT_PREC_all_lead_times.nc")

def sel_latlon(da, lat, lon):
    """Select nearest grid cell to a lat/lon point."""
    dist = np.sqrt((da.lat - lat)**2 + (da.lon - lon)**2)
    iy, ix = np.unravel_index(dist.values.argmin(), dist.shape)
    return da.isel(y=iy, x=ix)

# All ensemble members at Zürich across all lead times
zurich = sel_latlon(da, lat=47.37, lon=8.54)
print(zurich)  # shape: (eps=10, ref_time=1, lead_time=34)

# Single member, single hour (by index)
val = zurich.sel(eps=1).isel(lead_time=12)   # +12h, member 1
print(float(val))

# Mean over ensemble at +6h
print(float(zurich.isel(lead_time=6).mean("eps")))

<xarray.DataArray (eps: 10, ref_time: 1, lead_time: 34)> Size: 3kB
[340 values with dtype=float64]
Coordinates:
  * eps         (eps) int64 80B 1 2 3 4 5 6 7 8 9 10
  * ref_time    (ref_time) datetime64[ns] 8B 2026-04-05T06:00:00
  * lead_time   (lead_time) timedelta64[ns] 272B 00:00:00 ... 1 days 09:00:00
    valid_time  (ref_time, lead_time) datetime64[ns] 272B ...
    lon         float64 8B 8.55
    lat         float64 8B 47.37
Attributes:
    vref:         geo
    vcoord_type:  surface
    origin_z:     0.0
0.0
0.0


In [5]:
# Precipitation amount per hour, averaged over all members
hourly_mean = zurich.mean("eps").squeeze()  # shape: (lead_time=34,)
print(hourly_mean.values)  # mm per lead time step

# Which hours have any rain at all (any member > 0.1mm)?
rainy_hours = (zurich > 0.1).any("eps").squeeze()
print(rainy_hours.values)  # True/False for each of 34 hours

# Probability of rain per hour
prob_per_hour = (zurich > 0.1).mean("eps").squeeze() * 100
print(prob_per_hour.values)  # 0–100% for each hour

# Access a specific hour by offset (e.g. +18h)
h18 = zurich.isel(lead_time=18).mean("eps")
print(f"+18h mean precip: {float(h18):.3f} mm")

[0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 5.58296461e-04 1.13172936e-02 1.46862016e-02 6.59167441e-01
 6.59168463e-01 6.59168463e-01 7.22536600e-01 8.27824906e-01
 1.11472680e+00 1.12068344e+00 1.12068344e+00 1.12195124e+00
 1.12780464e+00 1.12780464e+00 1.12840930e+00 1.12841032e+00
 1.12841032e+00 1.12841032e+00 1.12841032e+00 1.12841032e+00
 1.12841032e+00 1.12841032e+00 1.12841032e+00 1.12841032e+00
 1.12841032e+00 1.12841032e+00]
[False False False False False False False False False False  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True]
[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0. 10. 50. 50. 50. 50. 50. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.]
+18h mean precip: 1.121 mm


In [7]:
# spezifischer zeit
prob_at_18h = (zurich.isel(lead_time=18) > 0.1).mean("eps") * 100
print(f"Rain probability at +18h: {float(prob_at_18h):.1f}%")

Rain probability at +18h: 60.0%


In [8]:
# Ensemble mean of accumulated precip at each lead time
mean_precip = zurich.mean("eps").squeeze()  # shape: (lead_time=34,)

# Hourly rain amount = difference between consecutive steps
hourly_rain = mean_precip.diff("lead_time")  # shape: (lead_time=33,)

# Rain in the 18th hour specifically (between +17h and +18h)
rain_at_18h = float(mean_precip.isel(lead_time=18) - mean_precip.isel(lead_time=17))
print(f"Rain in hour +18h: {rain_at_18h:.3f} mm/m²")

# Or see all hours at once
print(hourly_rain.values)

Rain in hour +18h: 0.000 mm/m²
[0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 5.58296461e-04
 1.07589971e-02 3.36890801e-03 6.44481239e-01 1.02155585e-06
 0.00000000e+00 6.33681375e-02 1.05288306e-01 2.86901890e-01
 5.95663938e-03 0.00000000e+00 1.26780927e-03 5.85339885e-03
 0.00000000e+00 6.04655322e-04 1.02155585e-06 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00]


In [9]:
import numpy as np

hourly_rain_vals = hourly_rain.values

# Round to 2 decimal places, zero out anything below 0.01 mm
hourly_rain_clean = np.where(hourly_rain_vals < 0.01, 0.0, np.round(hourly_rain_vals, 2))

for h, val in enumerate(hourly_rain_clean, start=1):
    print(f"+{h:02d}h → {val:.2f} mm/m²")

+01h → 0.00 mm/m²
+02h → 0.00 mm/m²
+03h → 0.00 mm/m²
+04h → 0.00 mm/m²
+05h → 0.00 mm/m²
+06h → 0.00 mm/m²
+07h → 0.00 mm/m²
+08h → 0.00 mm/m²
+09h → 0.01 mm/m²
+10h → 0.00 mm/m²
+11h → 0.64 mm/m²
+12h → 0.00 mm/m²
+13h → 0.00 mm/m²
+14h → 0.06 mm/m²
+15h → 0.11 mm/m²
+16h → 0.29 mm/m²
+17h → 0.00 mm/m²
+18h → 0.00 mm/m²
+19h → 0.00 mm/m²
+20h → 0.00 mm/m²
+21h → 0.00 mm/m²
+22h → 0.00 mm/m²
+23h → 0.00 mm/m²
+24h → 0.00 mm/m²
+25h → 0.00 mm/m²
+26h → 0.00 mm/m²
+27h → 0.00 mm/m²
+28h → 0.00 mm/m²
+29h → 0.00 mm/m²
+30h → 0.00 mm/m²
+31h → 0.00 mm/m²
+32h → 0.00 mm/m²
+33h → 0.00 mm/m²


In [10]:
for h, val in enumerate(hourly_rain_clean, start=1):
    if val > 0.0:
        print(f"+{h:02d}h → {val:.2f} mm/m²")

+09h → 0.01 mm/m²
+11h → 0.64 mm/m²
+14h → 0.06 mm/m²
+15h → 0.11 mm/m²
+16h → 0.29 mm/m²


In [3]:
import xarray as xr
from meteodatalab import ogd_api
from meteodatalab.operators import regrid
from rasterio.crs import CRS
from datetime import timedelta
from earthkit.data import config
import numpy as np

config.set("cache-policy", "temporary")

# --- Grid definition ---
extent = (-0.817, 18.183, 41.183, 51.183)
nx, ny = 429, 295
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)

# --- Fetch + regrid all lead times ---
das = []
for h in range(0, 34):
    req = ogd_api.Request(
        collection="ogd-forecasting-icon-ch1",
        variable="TOT_PREC",
        reference_datetime="latest",
        perturbed=True,
        horizon=timedelta(hours=h),
    )
    da_h = ogd_api.get_from_ogd(req)
    da_h_geo = regrid.iconremap(da_h, destination)
    das.append(da_h_geo)
    print(f"✓ +{h:02d}h")

# --- Concatenate ---
da_all = xr.concat(das, dim="lead_time")
da_all = da_all.assign_coords(lead_time=[timedelta(hours=h) for h in range(34)])

# --- Strip non-serializable attrs ---
def clean_attrs(attrs):
    valid_types = (str, int, float, bytes, list, tuple)
    return {k: v for k, v in attrs.items() if isinstance(v, valid_types)}

da_all.attrs = clean_attrs(da_all.attrs)
for coord in da_all.coords:
    da_all[coord].attrs = clean_attrs(da_all[coord].attrs)

# --- Compute hourly rain ---
mean_precip = da_all.mean("eps").squeeze("ref_time")        # (lead_time, y, x)
hourly_rain = mean_precip.diff("lead_time")                 # ← this line was missing!
hourly_rain = hourly_rain.drop_vars("valid_time", errors="ignore")
hourly_rain.values = np.where(hourly_rain.values < 0.01, 0.0, np.round(hourly_rain.values, 2))
hourly_rain.attrs = {"long_name": "Hourly precipitation (ensemble mean)", "units": "mm/m2"}

# --- Save ---
ds = xr.Dataset({
    "TOT_PREC": da_all,
    "hourly_rain": hourly_rain,
})
ds.to_netcdf("icon_ch1_TOT_PREC_all_lead_times.nc")
print("Saved!")

✓ +00h
✓ +01h
✓ +02h
✓ +03h
✓ +04h
✓ +05h
✓ +06h
✓ +07h
✓ +08h
✓ +09h
✓ +10h
✓ +11h
✓ +12h
✓ +13h
✓ +14h
✓ +15h
✓ +16h
✓ +17h
✓ +18h
✓ +19h
✓ +20h
✓ +21h
✓ +22h
✓ +23h
✓ +24h
✓ +25h
✓ +26h
✓ +27h
✓ +28h
✓ +29h
✓ +30h
✓ +31h
✓ +32h
✓ +33h
Saved!


In [5]:
ds = xr.open_dataset("icon_ch1_TOT_PREC_all_lead_times.nc")

# Access each variable
da_all = ds["TOT_PREC"]
hourly_rain = ds["hourly_rain"]

# --- Query by location ---
def sel_latlon(da, lat, lon):
    dist = np.sqrt((da.lat - lat)**2 + (da.lon - lon)**2)
    iy, ix = np.unravel_index(dist.values.argmin(), dist.shape)
    return da.isel(y=iy, x=ix)

# Hourly rain at Zürich
rain_zurich = sel_latlon(hourly_rain, lat=47.558, lon=7.587)

for h, val in enumerate(rain_zurich.values, start=1):
    if val > 0.0:
        print(f"+{h:02d}h → {val:.2f} mm/m²")

In [ ]:
lat=47.558, lon=7.587

In [8]:
ds = xr.open_dataset("icon_ch1_TOT_PREC_all_lead_times.nc")
da_all = ds["TOT_PREC"]
hourly_rain = ds["hourly_rain"]

def sel_latlon(da, lat, lon):
    dist = np.sqrt((da.lat - lat)**2 + (da.lon - lon)**2)
    iy, ix = np.unravel_index(dist.values.argmin(), dist.shape)
    return da.isel(y=iy, x=ix)

# --- Basel at +17h ---
rain_basel   = sel_latlon(hourly_rain, lat=47.56, lon=7.59)
tot_basel    = sel_latlon(da_all,      lat=47.56, lon=7.59)

rain_17h     = float(rain_basel.isel(lead_time=16))          # diff index 16 = between +16h and +17h
prob_17h     = float((
    (tot_basel.isel(lead_time=17) - tot_basel.isel(lead_time=16))
    .mean("eps") > 0.1
) * 100)  # simplified — use per-member diff for proper probability:
prob_17h     = float(((tot_basel.sel(eps=slice(1,10)).isel(lead_time=17) - 
                        tot_basel.sel(eps=slice(1,10)).isel(lead_time=16)) > 0.1).mean("eps") * 100)

print(f"Basel   +17h → rainfall: {rain_17h:.2f} mm/m²  |  probability: {prob_17h:.0f}%")

# --- Zürich at +30h ---
rain_zurich  = sel_latlon(hourly_rain, lat=47.37, lon=8.54)
tot_zurich   = sel_latlon(da_all,      lat=47.37, lon=8.54)

rain_30h     = float(rain_zurich.isel(lead_time=29))         # diff index 29 = between +29h and +30h
prob_30h     = float(((tot_zurich.sel(eps=slice(1,10)).isel(lead_time=30) - 
                        tot_zurich.sel(eps=slice(1,10)).isel(lead_time=29)) > 0.1).mean("eps") * 100)

print(f"Zürich  +30h → rainfall: {rain_30h:.2f} mm/m²  |  probability: {prob_30h:.0f}%")

Basel   +17h → rainfall: 0.00 mm/m²  |  probability: 0%
Zürich  +30h → rainfall: 0.00 mm/m²  |  probability: 0%


In [9]:
print(rain_30h)

0.0


In [13]:
# --- Find all cells with heavy rain (>5mm in any single hour) in next 33h ---
# hourly_rain shape: (lead_time=33, y=295, x=429)

THRESHOLD = 3.5  # mm — adjust to taste (2.0 = moderate, 5.0 = heavy, 10.0 = very heavy)

# Boolean mask: any hour exceeds threshold → shape (y, x)
heavy_mask = (hourly_rain > THRESHOLD).any(dim="lead_time")

# Get y/x indices of all matching cells
iy_all, ix_all = np.where(heavy_mask.values)

print(f"Found {len(iy_all)} cells with >{THRESHOLD}mm rain in any single hour\n")

for iy, ix in zip(iy_all, ix_all):
    lat = float(hourly_rain.lat.isel(y=iy, x=ix))
    lon = float(hourly_rain.lon.isel(y=iy, x=ix))
    
    # Which hours and how much?
    cell_rain = hourly_rain.isel(y=iy, x=ix)
    heavy_hours = [(h+1, float(v)) for h, v in enumerate(cell_rain.values) if v > THRESHOLD]
    hours_str = ", ".join([f"+{h:02d}h={v:.1f}mm" for h, v in heavy_hours])
    
    print(f"  lat={lat:.2f} lon={lon:.2f}  →  {hours_str}")

Found 13 cells with >3.5mm rain in any single hour

  lat=43.70 lon=2.82  →  +10h=3.8mm
  lat=45.77 lon=4.29  →  +11h=3.6mm
  lat=45.77 lon=4.33  →  +11h=3.6mm
  lat=45.77 lon=4.38  →  +11h=3.5mm
  lat=45.81 lon=4.38  →  +11h=4.3mm
  lat=47.65 lon=14.10  →  +17h=4.0mm
  lat=47.68 lon=13.30  →  +16h=3.8mm
  lat=47.68 lon=13.34  →  +16h=3.8mm
  lat=47.68 lon=14.10  →  +17h=3.7mm
  lat=47.68 lon=15.16  →  +18h=3.9mm
  lat=47.71 lon=14.10  →  +17h=5.0mm
  lat=47.71 lon=14.14  →  +17h=3.9mm
  lat=47.71 lon=14.99  →  +18h=4.2mm


In [15]:
pip install folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 451.2 kB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [22]:
import folium
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr

ds = xr.open_dataset("icon_ch1_TOT_PREC_all_lead_times.nc")
hourly_rain = ds["hourly_rain"]

# Max rain across all hours → shape (y, x)
max_rain = hourly_rain.max(dim="lead_time").values

# Normalize and colorize
norm = mcolors.Normalize(vmin=0, vmax=max_rain.max())
cmap = plt.cm.YlOrRd
rgba = cmap(norm(max_rain))                   # (y, x, 4)
rgba[..., 3] = np.where(max_rain < 0.01, 0, 0.65)  # transparent where no rain

# Flip vertically — image origin is top-left
rgba_flipped = np.flipud(rgba)

# Bounds from your grid extent
bounds = [[41.183, -0.817], [51.183, 18.183]]

m = folium.Map(location=[46.5, 8.5], zoom_start=7, tiles="OpenStreetMap")
folium.raster_layers.ImageOverlay(
    image=rgba_flipped,
    bounds=bounds,
    opacity=1.0,
    name="Max rain next 33h"
).add_to(m)
folium.LayerControl().add_to(m)
m  # displays inline in Jupyter
m.save("rain_map.html")

In [23]:
import numpy as np
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Distance mask ---
CH_LAT, CH_LON = 46.8, 8.2

lons = np.linspace(-0.817, 18.183, 429)
lats = np.linspace(41.183, 51.183, 295)
LON, LAT = np.meshgrid(lons, lats)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

dist_km = haversine(LAT, LON, CH_LAT, CH_LON)
within_mask = dist_km <= 350  # ~100km beyond Switzerland's border

# --- Colorize ---
norm = mcolors.Normalize(vmin=0, vmax=np.nanmax(max_rain))
cmap = plt.cm.YlOrRd
max_rain_masked = np.where(within_mask, max_rain, np.nan)
rgba = cmap(norm(np.nan_to_num(max_rain_masked)))
rgba[..., 3] = np.where(within_mask & (max_rain_masked >= 0.01), 0.65, 0)
rgba_flipped = np.flipud(rgba)

# --- Map ---
bounds = [[41.183, -0.817], [51.183, 18.183]]
m = folium.Map(location=[46.5, 8.5], zoom_start=7, tiles="CartoDB positron")
folium.raster_layers.ImageOverlay(
    image=rgba_flipped,
    bounds=bounds,
    opacity=0.6,
    name="Max rain next 33h"
).add_to(m)
folium.LayerControl().add_to(m)
m.save("rain_map2.html")
print("Saved!")

Saved!


In [24]:
import numpy as np
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Distance mask ---
CH_LAT, CH_LON = 46.8, 8.2

lons = np.linspace(-0.817, 18.183, 429)
lats = np.linspace(41.183, 51.183, 295)
LON, LAT = np.meshgrid(lons, lats)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

dist_km = haversine(LAT, LON, CH_LAT, CH_LON)
within_mask = dist_km <= 350

# --- Colorize ---
norm = mcolors.Normalize(vmin=0, vmax=np.nanmax(max_rain))
cmap = plt.cm.YlOrRd
max_rain_masked = np.where(within_mask, max_rain, np.nan)
rgba = cmap(norm(np.nan_to_num(max_rain_masked)))
rgba[..., 3] = np.where(within_mask & (max_rain_masked >= 0.01), 0.65, 0)
rgba_flipped = np.flipud(rgba)

# --- Map ---
bounds = [[41.183, -0.817], [51.183, 18.183]]

m = folium.Map(location=[46.5, 8.5], zoom_start=7, tiles=None)  # no default tiles

# 1. Base map — bottom
folium.TileLayer(
    tiles="CartoDB positron",
    name="Base map",
    overlay=False
).add_to(m)

# 2. Rain overlay — middle
folium.raster_layers.ImageOverlay(
    image=rgba_flipped,
    bounds=bounds,
    opacity=0.6,
    name="Max rain next 33h"
).add_to(m)

# 3. Labels only — top
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="CartoDB",
    name="Labels",
    overlay=True,
    zindex=900
).add_to(m)

folium.LayerControl().add_to(m)
m.save("rain_map3.html")
print("Saved!")

Saved!


In [25]:
import numpy as np
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --- Distance mask ---
CH_LAT, CH_LON = 46.8, 8.2

lons = np.linspace(-0.817, 18.183, 429)
lats = np.linspace(41.183, 51.183, 295)
LON, LAT = np.meshgrid(lons, lats)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

dist_km = haversine(LAT, LON, CH_LAT, CH_LON)
within_mask = dist_km <= 350

# --- Colorize ---
norm = mcolors.Normalize(vmin=0, vmax=np.nanmax(max_rain))
cmap = plt.cm.YlOrRd
max_rain_masked = np.where(within_mask, max_rain, np.nan)
rgba = cmap(norm(np.nan_to_num(max_rain_masked)))
rgba[..., 3] = np.where(within_mask & (max_rain_masked >= 0.01), 0.65, 0)
rgba_flipped = np.flipud(rgba)

# --- Map ---
bounds = [[41.183, -0.817], [51.183, 18.183]]

m = folium.Map(location=[46.5, 8.5], zoom_start=7, tiles=None)

# 1. Base map — bottom
folium.TileLayer(
    tiles="CartoDB positron",
    name="Base map",
    overlay=False
).add_to(m)

# 2. Rain overlay — middle
folium.raster_layers.ImageOverlay(
    image=rgba_flipped,
    bounds=bounds,
    opacity=0.6,
    name="Max rain next 33h"
).add_to(m)

# 3. Create a custom pane that sits above overlayPane (z=400)
m.get_root().html.add_child(folium.Element("""
    <script>
        document.addEventListener("DOMContentLoaded", function() {
            var map = Object.values(window).find(v => v && v._leaflet_id);
            var labelsPane = map.createPane('labelsPane');
            labelsPane.style.zIndex = 650;
            labelsPane.style.pointerEvents = 'none';
        });
    </script>
"""))

# 4. Labels on top pane
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="CartoDB",
    name="Labels",
    overlay=True,
    pane="labelsPane"
).add_to(m)

folium.LayerControl().add_to(m)
m.save("rain_map4.html")
print("Saved!")

Saved!
